In [13]:
!pip install underthesea pyvi

In [14]:
# Cell 1: Import thư viện và định nghĩa từ điển
from underthesea import word_tokenize
import re
from pprint import pprint

# Từ điển quy đổi khối lượng trung bình
avg_unit_mass = {
    # Rau củ quả
    "hành tây": {"củ": 150},
    "tỏi": {"củ": 40, "tép": 5},
    "gừng": {"củ": 80},
    "khoai tây": {"củ": 180},
    "cà chua": {"quả": 120},
    "chanh": {"quả": 70},
    "cam": {"quả": 250},
    "trứng gà": {"quả": 55},
    "ớt": {"quả": 8},
    "dưa leo": {"quả": 200},
    "cà rốt": {"củ": 100},
    "bắp cải": {"cây": 1000, "củ": 800},
    "xà lách": {"cây": 300},
    "bí đỏ": {"quả": 2000},
    "bí xanh": {"quả": 800},
    "đu đủ": {"quả": 1500},
    "dứa": {"quả": 900},
    "thơm": {"quả": 900},
    "chuối": {"quả": 120},

    # Đơn vị bó
    "rau muống": {"bó": 200},
    "rau cải": {"bó": 250},
    "rau ngót": {"bó": 200},
    "rau dền": {"bó": 200},
    "rau má": {"bó": 150},
    "rau lang": {"bó": 200},
    "rau mồng tơi": {"bó": 250},
    "cải xanh": {"bó": 250},
    "cải ngọt": {"bó": 250},
    "hành lá": {"bó": 100},
    "rau răm": {"bó": 100},
    "rau thơm": {"bó": 100},
    "tía tô": {"bó": 100},
    "kinh giới": {"bó": 100},
    "ngò rí": {"bó": 80},
    "ngò gai": {"bó": 80},
    "cần tây": {"bó": 300},
}

# Từ điển quy đổi cho đơn vị đo lường
weight_units = {
    "kg": 1000,
    "ký": 1000,
    "cân": 1000,
    "lạng": 100,
    "g": 1,
    "gram": 1
}

# Đơn vị đếm
count_units = {
    "miếng": 40,
    "khoanh": 60
}

In [15]:
# Cell 2: Danh sách nguyên liệu
veg = [
    "rau muống","rau cải","rau ngót","rau dền","rau má","rau lang","rau mồng tơi",
    "cải xanh","cải ngọt","cải thảo","bắp cải","cải xoăn","xà lách","diếp cá",
    "hành lá","hẹ","cần tây","cần nước","rau răm","rau thơm",
    "tía tô","kinh giới","ngò rí","ngò gai","lá lốt",
    "lá chanh","lá quế","đọt bí","bông bí","rau sam","hành tím","hành khô",
    "cà chua","dưa leo","dưa chuột","bí đỏ","bí xanh","bí đao",
    "mướp","khổ qua","cà tím","khoai tây"
]

meat = [
    "thịt bò","thịt heo","thịt lợn","thịt gà","thịt vịt","thịt dê","thịt cừu","thịt trâu","thịt ngan","thịt ngỗng","thịt thỏ",
    "sườn bò","bắp bò","gân bò","đuôi bò","tim bò","gan bò","lưỡi bò","óc bò",
    "sườn heo","sườn non","ba chỉ heo","ba rọi","nạc heo","mỡ heo","giò heo","chân giò","tai heo","đuôi heo","tim heo","gan heo","lòng heo","bao tử heo","cật heo","óc heo","phèo heo",
    "ức gà","đùi gà","cánh gà","chân gà","tim gà","gan gà","mề gà","cổ gà",
    "ức vịt","đùi vịt","cánh vịt","tim vịt","gan vịt","cổ vịt",
    "thịt xay","thịt băm",
    "tôm","cua","chả cua","ghẹ","mực","bạch tuộc",
    "cá hồi","cá thu","cá ngừ","cá basa",
    "cá rô","cá lóc","cá diêu hồng",
    "cá trê","cá cơm","cá chép",
    "sò","nghêu","hến","ốc",
    "lòng bò","lá lách bò","phổi bò","thực quản bò","bầu dục bò",
    "móng giò heo","đầu heo","mũi heo","má heo","lưỡi heo","huyết heo","tiết heo","mỡ heo non","bì heo","da heo","sụn heo","xương heo","xương ống heo","xương cục heo","lòng non heo","lòng già heo","dồi lòng","dạ dày heo",
    "lòng gà","mề vịt","da gà","da vịt","lòng ngan","mề ngan","gan ngan",
    "lòng dê","xương dê","đuôi cừu","sụn cừu",
    "đầu cá","xương cá","lườn cá","bụng cá","vây cá","đuôi cá","da cá","trứng cá","bao tử cá","gan cá","bong bóng cá","mỡ cá hồi",
    "đầu tôm","vỏ tôm","chân tôm","ruột tôm","mai mực","xương mực","lưỡi mực","thịt chân ốc","ruột ốc"
]

fruit = [
    "xoài","đu đủ","dứa","thơm","chuối",
    "táo","lê","cam","quýt","bưởi",
    "nho","dưa hấu","dưa lưới","thanh long",
    "vải","nhãn","mận"
]

spice = [
    "nước mắm","muối","đường","hạt nêm","bột ngọt",
    "tiêu","ớt","ớt bột","ớt tươi",
    "tỏi","gừng","sả","riềng","nghệ",
    "dầu ăn","dầu hào","nước tương","xì dầu",
    "hành tây"
]

starch = [
    "bánh phở", "bún", "miến", "mì", "mì tôm", "mì sợi",
    "nui", "macaroni", "spaghetti", "bánh đa", "bánh đa cua",
    "bánh canh", "hủ tiếu"
]

all_ing = set(veg + meat + fruit + spice + starch)

In [16]:
# Cell 3: Hàm xử lý tên món ăn
def get_dish(text):
    """Trích xuất tên món ăn"""
    text = text.lower()
    first = text.strip().split("\n")[0]

    if "nguyên liệu" in text:
        first = text.split("nguyên liệu")[0]

    tokens = word_tokenize(first, format="text")
    return tokens.strip()

def contains_phrase(text, phrase):
    """Kiểm tra cụm từ có trong văn bản"""
    pattern = r"\b" + re.escape(phrase) + r"\b"
    return re.search(pattern, text) is not None

def get_main(name, max_main=4):
    """Xác định nguyên liệu chính từ tên món ăn"""
    if not name:
        return []

    name = name.lower().strip()
    mains = []

    # Món ăn -> tinh bột
    dish_to_starch = {
        "phở": "bánh phở",
        "bún": "bún",
        "miến": "miến",
        "mì": "mì",
        "hủ tiếu": "hủ tiếu",
        "bánh canh": "bánh canh",
        "bánh đa": "bánh đa"
    }

    # Món ăn -> protein
    dish_to_protein = {
        "phở bò": "thịt bò",
        "phở gà": "thịt gà",
        "bún bò": "thịt bò",
        "bún riêu": "cua",
        "miến gà": "thịt gà",
        "cháo lòng": "lòng heo",
        "bún chả": "thịt heo",
        "mì quảng": "thịt heo",
        "phở cuốn": "thịt bò"
    }

    # 1. Tìm tinh bột
    for keyword, starch_item in sorted(dish_to_starch.items(), key=lambda x: len(x[0]), reverse=True):
        if contains_phrase(name, keyword):
            mains.append(starch_item)
            break

    # 2. Tìm protein chính
    for dish, protein in sorted(dish_to_protein.items(), key=lambda x: len(x[0]), reverse=True):
        if contains_phrase(name, dish):
            if protein not in mains:
                mains.append(protein)

    # 3. Tìm nguyên liệu khác
    all_items = meat + veg + fruit
    all_items = sorted(all_items, key=len, reverse=True)

    for item in all_items:
        if len(mains) >= max_main:
            break
        if contains_phrase(name, item):
            if item not in mains:
                mains.append(item)

    return mains[:max_main]

In [17]:
# Cell 4: Hàm chuyển đổi đơn vị
def convert_to_grams(value, unit, ingredient_name=None):
    """Chuyển đổi giá trị và đơn vị thành gram"""
    if not unit:
        return {"value": value, "unit": unit}

    unit = unit.lower()

    # Xử lý đơn vị trọng lượng
    if unit in weight_units:
        value *= weight_units[unit]
        return {"value": value, "unit": "g"}

    # Xử lý đơn vị đếm đặc biệt (củ, quả, tép, cây, bó)
    elif unit in ["củ", "quả", "tép", "cây", "bó"]:
        if ingredient_name and ingredient_name in avg_unit_mass:
            if unit in avg_unit_mass[ingredient_name]:
                avg_mass = avg_unit_mass[ingredient_name][unit]
                value *= avg_mass
                return {"value": value, "unit": "g"}

    # Xử lý đơn vị đếm có sẵn (miếng, khoanh)
    elif unit in count_units:
        value *= count_units[unit]
        return {"value": value, "unit": "g"}

    # Không thể chuyển đổi, giữ nguyên
    return {"value": value, "unit": unit}

def parse_quantity(quantity_text, ingredient_name=None):
    """Parse chuỗi số lượng và đơn vị"""
    if not quantity_text:
        return None

    quantity_text = quantity_text.lower().strip()

    # Pattern cho số thập phân
    pattern = r"(\d+(?:\.\d+)?)\s*([a-zàáãạảăắằặẵâấầậẫẩđêễệểếéèẻẽẹóòỏõọôốồổỗộơớờởỡợíìỉĩịúùủũụưứừửữựýỳỷỹỵ]+)?"

    match = re.search(pattern, quantity_text)
    if not match:
        return None

    try:
        value = float(match.group(1))
    except ValueError:
        return None

    unit = match.group(2) if match.group(2) else None

    if unit:
        return convert_to_grams(value, unit, ingredient_name)

    return {"value": value, "unit": None}

In [18]:
# Cell 5: Hàm tìm nguyên liệu và số lượng
def find_ingredients_with_quantity(text):
    """Tìm tất cả nguyên liệu và số lượng trong văn bản"""
    if not text:
        return []

    text = text.lower()
    matches = []

    # Pattern cho số lượng
    quantity_pattern = r"(\d+(?:\.\d+)?)\s*([a-zàáãạảăắằặẵâấầậẫẩđêễệểếéèẻẽẹóòỏõọôốồổỗộơớờởỡợíìỉĩịúùủũụưứừửữựýỳỷỹỵ]+)?"

    # Sắp xếp nguyên liệu theo độ dài giảm dần
    sorted_ing = sorted(all_ing, key=len, reverse=True)

    for ing in sorted_ing:
        pattern = r"\b" + re.escape(ing) + r"\b"

        for match in re.finditer(pattern, text):
            quantity_text = None

            # Tìm số lượng đằng trước
            before_start = max(0, match.start() - 20)
            before_text = text[before_start:match.start()]
            quantity_match = re.search(quantity_pattern + r"\s*$", before_text)

            if quantity_match:
                quantity_text = quantity_match.group(0).strip()
            else:
                # Tìm số lượng đằng sau
                after_end = min(len(text), match.end() + 20)
                after_text = text[match.end():after_end]
                quantity_match = re.search(r"^\s*" + quantity_pattern, after_text)
                if quantity_match:
                    quantity_text = quantity_match.group(0).strip()

            matches.append({
                "ingredient": ing,
                "quantity_text": quantity_text,
                "start": match.start(),
                "end": match.end(),
                "length": len(ing)
            })

    # Xử lý overlap
    matches.sort(key=lambda x: (x["start"], -x["length"]))

    filtered = []
    occupied = []

    for m in matches:
        overlap = False
        for s, e in occupied:
            if not (m["end"] <= s or m["start"] >= e):
                overlap = True
                break

        if not overlap:
            filtered.append((m["ingredient"], m["quantity_text"], m["start"]))
            occupied.append((m["start"], m["end"]))

    return filtered

# Mapping tên con vật (ngắn) -> danh sách các biến thể nguyên liệu
animal_variants = {
    "cua": ["cua", "chả cua", "ghẹ"],
    "bò": ["thịt bò", "sườn bò", "bắp bò", "gân bò", "đuôi bò", "tim bò", "gan bò",
           "lưỡi bò", "óc bò", "lòng bò", "lá lách bò", "phổi bò", "thực quản bò", "bầu dục bò"],
    "heo": ["thịt heo", "thịt lợn", "sườn heo", "sườn non", "ba chỉ heo", "ba rọi",
            "nạc heo", "mỡ heo", "giò heo", "chân giò", "tai heo", "đuôi heo", "tim heo",
            "gan heo", "lòng heo", "bao tử heo", "cật heo", "óc heo", "phèo heo",
            "móng giò heo", "đầu heo", "mũi heo", "má heo", "lưỡi heo", "huyết heo",
            "tiết heo", "mỡ heo non", "bì heo", "da heo", "sụn heo", "xương heo",
            "xương ống heo", "xương cục heo", "lòng non heo", "lòng già heo", "dồi lòng",
            "dạ dày heo"],
    "lợn": ["thịt heo", "thịt lợn", "sườn heo", "sườn non", "ba chỉ heo", "ba rọi",
            "nạc heo", "mỡ heo", "giò heo", "chân giò", "tai heo", "đuôi heo", "tim heo",
            "gan heo", "lòng heo", "bao tử heo", "cật heo", "óc heo", "phèo heo"],
    "gà": ["thịt gà", "ức gà", "đùi gà", "cánh gà", "chân gà", "tim gà", "gan gà",
           "mề gà", "cổ gà", "lòng gà", "da gà"],
    "vịt": ["thịt vịt", "ức vịt", "đùi vịt", "cánh vịt", "tim vịt", "gan vịt",
            "cổ vịt", "mề vịt", "da vịt"],
    "ngan": ["thịt ngan", "lòng ngan", "mề ngan", "gan ngan"],
    "dê": ["thịt dê", "lòng dê", "xương dê"],
    "cừu": ["thịt cừu", "đuôi cừu", "sụn cừu"],
    "tôm": ["tôm", "đầu tôm", "vỏ tôm", "chân tôm", "ruột tôm"],
    "mực": ["mực", "bạch tuộc", "mai mực", "xương mực", "lưỡi mực"],
    "cá": ["cá hồi", "cá thu", "cá ngừ", "cá basa", "cá rô", "cá lóc", "cá diêu hồng",
           "cá trê", "cá cơm", "cá chép", "đầu cá", "xương cá", "lườn cá", "bụng cá",
           "vây cá", "đuôi cá", "da cá", "trứng cá", "bao tử cá", "gan cá", "bong bóng cá",
           "mỡ cá hồi"],
    "ốc": ["ốc", "sò", "nghêu", "hến", "thịt chân ốc", "ruột ốc"]
}

In [28]:
# Cell 6: Hàm extract chính
def is_main_ingredient(ing, mains):
    """Kiểm tra nguyên liệu có phải main dựa trên tên món ăn (chỉ cần trùng tên con vật)"""
    if ing in mains:
        return True

    for animal, variants in animal_variants.items():
        if animal in mains:
            if ing in variants:
                return True

    return False

def extract(name, ingredients_text, include_spice_quantity=True):
    """Trích xuất danh sách nguyên liệu từ tên món ăn và văn bản nguyên liệu

    Args:
        name: Tên món ăn
        ingredients_text: Văn bản chứa nguyên liệu
        include_spice_quantity:
            - True  -> giữ số lượng/đơn vị của gia vị
            - False -> bỏ số lượng/đơn vị của gia vị
    """

    if not ingredients_text:
        return []

    text = ingredients_text.lower()
    mains = get_main(name)

    out = []
    ingredients_found = find_ingredients_with_quantity(text)

    for ing, quantity_text, _ in ingredients_found:
        item = {
            "ingredient": ing,
            "value": None,
            "unit": None,
            "type": None
        }

        # Phân loại nguyên liệu trước
        if ing in spice:
            item["type"] = "optional"
        elif is_main_ingredient(ing, mains):
            item["type"] = "main"
        else:
            item["type"] = "required"

        # Nếu là spice và không muốn lưu quantity
        if item["type"] == "optional" and not include_spice_quantity:
            pass
        else:
            # Parse số lượng nếu có
            if quantity_text:
                parsed = parse_quantity(quantity_text, ing)
                if parsed:
                    item["value"] = parsed["value"]
                    item["unit"] = parsed["unit"]

        out.append(item)

    return out

def group(data):
    """Nhóm nguyên liệu theo loại"""
    # Method 1: Create empty dict and add in desired order
    res = {}
    res["main"] = []
    res["required"] = []   # required SECOND
    res["optional"] = []   # optional THIRD

    for x in data:
        res[x["type"]].append(x)

    return res

In [32]:
# Cell 7: Test chương trình
name = "Bún cua"

ingredients = """
0.5kg bún tươi, 3 lạng thịt bò, 2 khoanh giò heo, 0.2kg chả cua,
1 bó rau muống, 1 bó rau răm, 1 củ hành tây, 3 cây sả,
1 miếng riềng, 2 muỗng mắm ruốc, 1 muỗng ớt bột, 1 muỗng dầu ăn,
2 muỗng nước mắm, 1 muỗng đường, 1 ít muối, 2 quả chanh, ớt tươi
"""

result = group(extract(name, ingredients))
pprint(result, sort_dicts=False)

{'main': [{'ingredient': 'bún', 'value': 500.0, 'unit': 'g', 'type': 'main'},
          {'ingredient': 'chả cua',
           'value': 200.0,
           'unit': 'g',
           'type': 'main'}],
 'required': [{'ingredient': 'thịt bò',
               'value': 300.0,
               'unit': 'g',
               'type': 'required'},
              {'ingredient': 'giò heo',
               'value': 120.0,
               'unit': 'g',
               'type': 'required'},
              {'ingredient': 'rau muống',
               'value': 200.0,
               'unit': 'g',
               'type': 'required'},
              {'ingredient': 'rau răm',
               'value': 100.0,
               'unit': 'g',
               'type': 'required'}],
 'optional': [{'ingredient': 'hành tây',
               'value': 150.0,
               'unit': 'g',
               'type': 'optional'},
              {'ingredient': 'sả',
               'value': 3.0,
               'unit': 'cây',
               'type': 'optional'

In [34]:
# Convert to desired JSON format
import json

def convert_to_json_format(dish_name, grouped_data):
    """Convert grouped result to JSON format"""
    output = {
        "name": dish_name,
        "main_ingredients": {},
        "required_ingredients": {},
        "optional_ingredients": []  # Array for optional ingredients
    }

    # Main ingredients as key-value pairs
    for item in grouped_data.get("main", []):
        ingredient = item["ingredient"]
        quantity = item["value"] if item["value"] is not None else 1
        output["main_ingredients"][ingredient] = quantity

    # Required ingredients as key-value pairs
    for item in grouped_data.get("required", []):
        ingredient = item["ingredient"]
        quantity = item["value"] if item["value"] is not None else 1
        output["required_ingredients"][ingredient] = quantity

    # Optional ingredients as array
    for item in grouped_data.get("optional", []):
        ingredient = item["ingredient"]
        output["optional_ingredients"].append(ingredient)

    return output

# Convert result to JSON format
json_data = convert_to_json_format(name, result)

# Print as JSON string
print("\n" + "="*50)
print("JSON Output:")
print("="*50)
print(json.dumps(json_data, ensure_ascii=False, indent=2))



JSON Output:
{
  "name": "Bún cua",
  "main_ingredients": {
    "bún": 500.0,
    "chả cua": 200.0
  },
  "required_ingredients": {
    "thịt bò": 300.0,
    "giò heo": 120.0,
    "rau muống": 200.0,
    "rau răm": 100.0
  },
  "optional_ingredients": [
    "hành tây",
    "sả",
    "riềng",
    "ớt bột",
    "dầu ăn",
    "nước mắm",
    "đường",
    "muối",
    "ớt tươi"
  ]
}
